In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Authenticated ✓")

# 🩺 KasaMed — Prescription Explainer for Ghana (Gemma 4, offline)

**Build with Gemma: Ghana** — Healthcare track (GenAI for Good · Edge/On-Device · Multimodal)

## The problem
A patient leaves a Ghanaian clinic or pharmacy with a handwritten prescription or a
box of pills labelled in English medical shorthand — e.g. `amoxicillin 500mg TID x 5/7`.
Low literacy, English as a second language, and long distances to the nearest pharmacist
mean people mis-dose, stop antibiotics early, or skip medication entirely.

## The solution
Point a phone camera at the prescription or medicine packet. **Gemma 4 (E4B)** reads it
(vision), extracts the drug / dose / frequency / duration, then explains it in **plain Twi**
with a simple dosing schedule — **fully offline** on a low-end device.

## Why Gemma 4 E4B
- **E4B is an on-device edge model** (~5 GB) that accepts **image + text** input, so the
  offline claim is genuine, not aspirational.
- Runs on this free Kaggle GPU for the demo, and on a phone via LiteRT / AI Edge Gallery.
- Open weights + local inference = **patient data never leaves the device** and **no per-call cost**.

> ⚠️ **Confirm exact IDs against the model card** at
> https://www.kaggle.com/models/google/gemma-4 and https://ai.google.dev/gemma/docs/core .
> Gemma 4 is new; the repo string / processor class names below are the expected pattern
> (matching Gemma 3's multimodal API) but should be verified when you run this.

## 1 · Setup
Install dependencies. On Kaggle, enable a GPU accelerator (Settings → Accelerator → GPU).

In [ ]:
# Kaggle usually has torch + transformers pre-installed. Upgrade transformers for Gemma 4 support.
!pip install -q -U transformers accelerate pillow
# Optional: kagglehub to pull the model straight from the Kaggle model hub
!pip install -q -U kagglehub

## 2 · Load Gemma 4 E4B (vision + text)

Two ways to get the weights — pick whichever authenticates cleanly in your notebook:
1. **Hugging Face** (needs `HF_TOKEN` secret + license accepted), or
2. **Kaggle Models** via `kagglehub` (works inside Kaggle with no extra token).

We load the *instruction-tuned* multimodal checkpoint.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# --- CONFIRM THIS STRING against the model card before running ---
MODEL_ID = "google/gemma-4-e4b-it"   # expected HF pattern; E4B = on-device edge model

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print("Loaded:", MODEL_ID, "| device:", model.device)

## 3 · Core helper: `ask_gemma(image, prompt)`

One thin wrapper around the chat/multimodal API. Pass an optional PIL image plus a text
prompt; get text back. Everything else is built on this.

In [ ]:
from PIL import Image

def ask_gemma(prompt: str, image: Image.Image | None = None, max_new_tokens: int = 512) -> str:
    """Run one multimodal turn through Gemma 4. Image is optional."""
    content = []
    if image is not None:
        content.append({"type": "image", "image": image})
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    # Only decode the newly generated tokens, not the prompt
    return processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

## 4 · Stage 1 — Vision extraction to structured JSON

We deliberately split the pipeline into two stages. Stage 1 does **only** perception:
read the prescription image and return a strict JSON record. Keeping extraction separate
from explanation makes the system easier to debug, easier to show in the writeup, and much
more reliable than asking for everything in one shot.

**Safety by design:** the model must copy only what it can actually read. Anything unclear
becomes `"unclear"` — it must never guess a drug name or a dose.

In [ ]:
import json, re

EXTRACTION_PROMPT = """You are a careful medical transcription assistant.
Look at this image of a prescription or medicine packet.
Extract ONLY what is clearly written. Do NOT guess or invent anything.
If a field is unreadable or absent, use the string "unclear".

Return ONLY valid JSON (no markdown, no commentary) with this exact schema:
{
  "medications": [
    {
      "name": "drug name as written",
      "strength": "e.g. 500mg or unclear",
      "form": "tablet/capsule/syrup/injection/unclear",
      "frequency": "how often, e.g. 3 times a day / TID / unclear",
      "duration": "e.g. 5 days / 5/7 / unclear",
      "instructions": "any extra notes e.g. after food / unclear"
    }
  ],
  "overall_legible": true
}
Set "overall_legible" to false if the image is too blurry or handwriting is unreadable."""

def extract_prescription(image: Image.Image) -> dict:
    raw = ask_gemma(EXTRACTION_PROMPT, image=image, max_new_tokens=512)
    # Be tolerant if the model wraps JSON in ```json fences
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    text = m.group(0) if m else raw
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"medications": [], "overall_legible": False, "_raw": raw}

## 5 · Stage 2 — Plain-Twi explanation + dosing schedule

Stage 2 takes the clean JSON and produces the patient-facing output in the chosen language
(Twi by default; Ewe / Ga / English selectable). It expands shorthand into concrete timing
("morning, afternoon, night"), and it is instructed to stay strictly within the prescription —
no new medical advice — and to always tell the patient to confirm with a pharmacist/nurse.

In [ ]:
LANG_CHOICES = {
    "twi": "Twi (Akan)",
    "ewe": "Ewe",
    "ga":  "Ga",
    "en":  "simple English",
}

def explain_in_language(record: dict, lang: str = "twi") -> str:
    lang_name = LANG_CHOICES.get(lang, "Twi (Akan)")
    prompt = f"""You are a friendly community pharmacist speaking to a patient in {lang_name}.
Here is the prescription data (JSON):
{json.dumps(record, ensure_ascii=False, indent=2)}

Write a short, warm explanation IN {lang_name} that a person with low literacy can follow.
For EACH medicine include:
- what it is for (only if obvious from the drug; otherwise say to ask the nurse)
- exactly WHEN to take it, turned into concrete daily times (morning / afternoon / evening / night)
- how many days to continue, and to finish the full course if it is an antibiotic
- any instruction like "after food"

RULES:
- Use ONLY the information given. Do NOT add new medicines, doses, or medical claims.
- If a field is "unclear" or overall_legible is false, clearly say that part could not be read
  and advise showing the paper to a pharmacist or nurse.
- End with one short sentence reminding them to confirm with a pharmacist or nurse.
Keep it simple and kind."""
    return ask_gemma(prompt, image=None, max_new_tokens=768)

## 6 · End-to-end pipeline

In [ ]:
DISCLAIMER = {
    "twi": "⚠️ Saa mmoa yi firi krataa a wɔakyerɛw no. Kɔ pharmacist anaa nɛɛse hɔ na wɔmmoa wo.",
    "en":  "⚠️ This help is based only on the written paper. Please confirm with a pharmacist or nurse.",
}

def kasamed(image: Image.Image, lang: str = "twi") -> dict:
    record = extract_prescription(image)
    explanation = explain_in_language(record, lang=lang)
    return {
        "structured": record,
        "explanation": explanation,
        "disclaimer": DISCLAIMER.get(lang, DISCLAIMER["en"]),
    }

## 7 · Demo

Upload a photo of a prescription or medicine box to the notebook (or use a Kaggle dataset),
set the path below, and run. For the pitch, try a real pill packet.

In [ ]:
# --- point this at your test image ---
IMAGE_PATH = "//kaggle/input/datasets/niikofiannangquaye/kasamed-test-images"  # <-- change me

img = Image.open(IMAGE_PATH).convert("RGB")
img.thumbnail((896, 896))  # keep it light for edge-realistic conditions

result = kasamed(img, lang="twi")

print("=== STRUCTURED (Stage 1) ===")
print(json.dumps(result["structured"], ensure_ascii=False, indent=2))
print("\n=== TWI EXPLANATION (Stage 2) ===")
print(result["explanation"])
print("\n" + result["disclaimer"])

## 8 · The offline / on-device story (for the pitch & writeup)

This notebook runs E4B on a Kaggle GPU for convenience, but **E4B is the actual edge model**.
To prove offline operation for judges:

- **Google AI Edge Gallery** (Android) runs Gemma 4 E4B/E2B locally — load the same prompts,
  switch the phone to **airplane mode**, and it still works. That is the demo money-shot.
- Production path: **LiteRT / MediaPipe** on Android, or **ML Kit GenAI Prompt API**.
- **Why local matters here:** patient health data never leaves the phone (privacy), it works
  with **no internet** in rural clinics, and there is **zero per-inference cost**.

### Honest limitations to state in the writeup
- Handwriting OCR is the hard part; the `"unclear"` + "ask a pharmacist" guardrail is a
  deliberate safety choice, not a bug.
- Twi medical vocabulary may need a small glossary / few-shot examples to sharpen — a clear
  "what we'd do next" line for judges.
- This is a decision-support aid, **not** a medical device.